# 0AD-Bench v0.1

This notebook inspects and validates the 50-task benchmark catalog, runs a deterministic smoke episode through the complete harness, calculates a reference score, writes the standard trajectory layout, and produces a leaderboard row.

The smoke environment verifies benchmark plumbing without downloading the game. Engine-backed evaluation uses 0 A.D. Release 28 and the `ZeroADEnvironment` adapter shown in the final section.

In [ ]:
# @title 1. Install 0AD-Bench from the benchmark branch
%pip install -q "zero-ad-bench[validation] @ git+https://github.com/ritwikraha/markov-chainsaw.git@0ad-bench-v0.1#subdirectory=0ad-bench"

In [ ]:
# @title 2. Imports and output directory
import json, pathlib, shutil
import pandas as pd
from IPython.display import Markdown, display
from zero_ad_bench import BenchmarkRunner, TaskRegistry, aggregate_leaderboard, leaderboard_markdown
from zero_ad_bench.smoke import EconomySmokeEnvironment, SmokeAgent

OUTPUT_ROOT = pathlib.Path("/content/0ad-bench-results")
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True)
print("Output directory:", OUTPUT_ROOT)

In [ ]:
# @title 3. Load and summarize the v0.1 task catalog
registry = TaskRegistry.default()
summary = registry.summary()
print(json.dumps(summary, indent=2))
assert summary["task_count"] == 50
assert set(summary["categories"].values()) == {10}
tasks = pd.DataFrame([{"id": task.id, "split": task.split, "category": task.category, "tracks": ", ".join(task.tracks), "title": task.title} for task in registry.list()])
display(tasks)

In [ ]:
# @title 4. Inspect the experiment matrix
matrix = tasks.groupby(["category", "split"]).size().unstack(fill_value=0)
display(matrix)
display(tasks.groupby("category")["tracks"].apply(lambda values: sum("vision" in value for value in values)).rename("vision_tasks").to_frame())

In [ ]:
# @title 5. Inspect one adaptive task
task = registry.get("adapt-010")
print(json.dumps(task.to_dict(), indent=2))

In [ ]:
# @title 6. Run a complete deterministic smoke episode
smoke_task = registry.get("econ-001")
episode = BenchmarkRunner(OUTPUT_ROOT).run(
    smoke_task, SmokeAgent(), EconomySmokeEnvironment(),
    episode_id="smoke-econ-001",
)
print(json.dumps({"success": episode["success"], "metrics": episode["metrics"], "score": episode["score"]}, indent=2))

In [ ]:
# @title 7. Verify the trajectory contract
episode_dir = OUTPUT_ROOT / "smoke-econ-001"
artifacts = sorted(str(path.relative_to(episode_dir)) for path in episode_dir.rglob("*") if path.is_file())
print("\n".join(artifacts[:12]))
print(f"... {len(artifacts)} files total")
required = {"metadata.json", "actions.jsonl", "reasoning.jsonl", "rewards.jsonl", "summary.json"}
assert required.issubset(set(artifacts))
assert any(name.startswith("observations/") for name in artifacts)

In [ ]:
# @title 8. Produce a leaderboard row
rows = aggregate_leaderboard([episode])
display(pd.DataFrame(rows))
display(Markdown(leaderboard_markdown(rows)))

## Connect the real 0 A.D. engine

Use the official Release 28 RL client and supply the running client, action module, scenario factory, replay exporter, and deterministic perturbation hook to the adapter.

```python
from zero_ad_bench.adapters import ZeroADEnvironment

environment = ZeroADEnvironment(
    game=game,
    actions_module=zero_ad.actions,
    reset_payload_factory=build_scenario,
    perturbation_hook=apply_fixture_perturbation,
    replay_exporter=export_native_replay,
    close_hook=engine.stop,
)
summary = BenchmarkRunner(OUTPUT_ROOT).run(registry.get("econ-001"), agent, environment)
```

The existing Age of LLMs notebooks contain the verified Release 28 process launcher, RL connection, replay saving, and native replay video capture.